# 8. Predictive Analysis — Machine Learning

Train and evaluate Logistic Regression, SVM, Decision Tree, and KNN.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline


## 8.1 Load & Prepare Data

In [ ]:
df = pd.read_csv('../data/dataset_part_2.csv')

# One-hot encode categoricals
features = ['PayloadMass','Orbit','LaunchSite','Flights','GridFins',
            'Reused','Legs','LandingPad','Block','ReusedCount','Serial']

X = pd.get_dummies(df[features])
print('Feature matrix shape:', X.shape)

# TASK 1 — Create Y
Y = df['Class'].to_numpy()
print('Label shape:', Y.shape)
print('Success rate: {:.1f}%'.format(Y.mean()*100))


## 8.2 Standardize & Split

In [ ]:
# TASK 2 — Standardize
transform = StandardScaler()
X = transform.fit_transform(X)

# TASK 3 — Train/Test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2
)
print('Train:', X_train.shape, '  Test:', X_test.shape)


## 8.3 Logistic Regression

In [ ]:
# TASK 4
parameters_lr = {'C':[0.01,0.1,1],'penalty':['l2'],'solver':['lbfgs']}
lr = LogisticRegression(max_iter=1000)
logreg_cv = GridSearchCV(lr, parameters_lr, cv=10)
logreg_cv.fit(X_train, Y_train)
print('Best params:', logreg_cv.best_params_)
print('Best CV score:', round(logreg_cv.best_score_, 4))


In [ ]:
# TASK 5 — Test accuracy
print('LR Test Accuracy:', round(logreg_cv.score(X_test, Y_test), 4))


## 8.4 Support Vector Machine

In [ ]:
# TASK 6
parameters_svm = {
    'kernel':('linear','rbf','poly','sigmoid'),
    'C':np.logspace(-3,3,5),
    'gamma':np.logspace(-3,3,5)
}
svm = SVC()
svm_cv = GridSearchCV(svm, parameters_svm, cv=10)
svm_cv.fit(X_train, Y_train)
print('Best params:', svm_cv.best_params_)
print('Best CV score:', round(svm_cv.best_score_, 4))


In [ ]:
# TASK 7 — Test accuracy
print('SVM Test Accuracy:', round(svm_cv.score(X_test, Y_test), 4))


## 8.5 Decision Tree

In [ ]:
# TASK 8
parameters_tree = {
    'criterion':['gini','entropy'],
    'splitter':['best','random'],
    'max_depth':[2,4,6,8,10],
    'max_features':['sqrt','log2'],
    'min_samples_leaf':[1,2,4],
    'min_samples_split':[2,5,10]
}
tree = DecisionTreeClassifier()
tree_cv = GridSearchCV(tree, parameters_tree, cv=10)
tree_cv.fit(X_train, Y_train)
print('Best params:', tree_cv.best_params_)
print('Best CV score:', round(tree_cv.best_score_, 4))


In [ ]:
# TASK 9 — Test accuracy
print('Decision Tree Test Accuracy:', round(tree_cv.score(X_test, Y_test), 4))


## 8.6 K-Nearest Neighbors

In [ ]:
# TASK 10
parameters_knn = {
    'n_neighbors': list(range(1,11)),
    'algorithm':['auto','ball_tree','kd_tree','brute'],
    'p':[1,2]
}
KNN = KNeighborsClassifier()
knn_cv = GridSearchCV(KNN, parameters_knn, cv=10)
knn_cv.fit(X_train, Y_train)
print('Best params:', knn_cv.best_params_)
print('Best CV score:', round(knn_cv.best_score_, 4))


In [ ]:
# TASK 11 — Test accuracy
print('KNN Test Accuracy:', round(knn_cv.score(X_test, Y_test), 4))


## 8.7 Confusion Matrix — Best Model

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Failure','Success'],
                yticklabels=['Failure','Success'])
    plt.title(title)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(Y_test, tree_cv.predict(X_test), 'Decision Tree Confusion Matrix')


## 8.8 Model Comparison Summary

In [ ]:
results = pd.DataFrame({
    'Model':  ['Logistic Regression','SVM','Decision Tree','KNN'],
    'CV Accuracy':   [round(logreg_cv.best_score_,4),
                      round(svm_cv.best_score_,4),
                      round(tree_cv.best_score_,4),
                      round(knn_cv.best_score_,4)],
    'Test Accuracy': [round(logreg_cv.score(X_test,Y_test),4),
                      round(svm_cv.score(X_test,Y_test),4),
                      round(tree_cv.score(X_test,Y_test),4),
                      round(knn_cv.score(X_test,Y_test),4)],
})

results = results.sort_values('Test Accuracy', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

best = results.iloc[0]
print(f'\n🏆 Best model: {best["Model"]} — Test Accuracy: {best["Test Accuracy"]:.4f}')


In [ ]:
# Bar chart comparison
x = results['Model']
w = 0.35
fig, ax = plt.subplots(figsize=(9,5))
bars1 = ax.bar(np.arange(len(x))-w/2, results['CV Accuracy'],   w, label='CV Accuracy',   color='steelblue')
bars2 = ax.bar(np.arange(len(x))+w/2, results['Test Accuracy'], w, label='Test Accuracy', color='tomato')
ax.set_xticks(np.arange(len(x)))
ax.set_xticklabels(x, rotation=15, ha='right')
ax.set_ylim(0.7, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison — CV vs Test Accuracy')
ax.legend()
for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f'{bar.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()
